In [1]:
%load_ext autoreload
%autoreload 2

from pathlib import Path

import numpy as np
import pandas as pd
import plotly.express as px

from cryptosense.data import load_processed
from cryptosense.features import add_features
from cryptosense.regimes import add_regimes
from cryptosense.setups import setup_flag
from cryptosense.backtest import run_backtest, summarize_backtest

DATA_DIR = Path.cwd().parent / "data"
df = add_regimes(add_features(load_processed("BTC/USDT", DATA_DIR)))

In [2]:
dev = df.loc[:"2023-12-31"].dropna(subset=["dist_sma_100", "rel_volume_20", "er_20"])
COST = 0.0015

rows = []
for setup in ["continuation", "pullback"]:
    signal = setup_flag(dev, setup)
    for hold in [5, 10]:
        trades, equity = run_backtest(dev, signal, hold, COST)
        rows.append({"strategy": f"{setup} {hold}d", **summarize_backtest(trades, equity)})

bh_signal = pd.Series(False, index=dev.index)
bh_signal.iloc[0] = True
trades, equity = run_backtest(dev, bh_signal, len(dev) - 2, COST)
rows.append({"strategy": "buy and hold", **summarize_backtest(trades, equity)})

results = pd.DataFrame(rows).set_index("strategy")
results.round(3)

,total_return,annual_volatility,max_drawdown,trades,time_in_market,win_rate,avg_trade
strategy,,,,,,,
continuation 5d,1.303,0.190,-0.341,32,0.072,0.750,0.029
continuation 10d,1.895,0.272,-0.344,24,0.108,0.583,0.055
pullback 5d,-0.346,0.351,-0.638,86,0.193,0.500,-0.001
pullback 10d,0.178,0.356,-0.573,55,0.247,0.491,0.008
buy and hold,4.162,0.727,-0.832,1,1.000,1.000,4.162


In [3]:
rng = np.random.default_rng(0)
N_RUNS = 2000

def net_trade_returns(frame, hold, cost):
    """Net return of a trade signalled on each day: buy next open, sell `hold` days later."""
    entry = frame["open"].shift(-1)
    exit_ = frame["open"].shift(-(1 + hold))
    return ((exit_ * (1 - cost)) / (entry * (1 + cost)) - 1).dropna()

rows = []
for hold in [5, 10]:
    trades, _ = run_backtest(dev, setup_flag(dev, "continuation"), hold, COST)
    actual, k = trades["net_return"].mean(), len(trades)

    all_days = net_trade_returns(dev, hold, COST)
    up_days = all_days[dev.loc[all_days.index, "trend"] == "up"]

    for label, pool in [("any day", all_days), ("up-trend day", up_days)]:
        sims = np.array([
            rng.choice(pool.to_numpy(), size=k, replace=False).mean()
            for _ in range(N_RUNS)
        ])
        rows.append({
            "hold": hold,
            "random pool": label,
            "trades": k,
            "actual_avg_trade": actual,
            "random_median": np.median(sims),
            "share_random_beating_actual": (sims >= actual).mean(),
        })

pd.DataFrame(rows).round(4)

,hold,random pool,trades,actual_avg_trade,random_median,share_random_beating_actual
0,5,any day,32,0.0295,0.0045,0.0490
1,5,up-trend day,32,0.0295,0.0146,0.1585
2,10,any day,24,0.0546,0.0119,0.0465
3,10,up-trend day,24,0.0546,0.0320,0.1945


In [4]:
rows = []
for cost in [0.0, 0.0015, 0.003]:
    for hold in [5, 10]:
        trades, equity = run_backtest(dev, setup_flag(dev, "continuation"), hold, cost)
        summary = summarize_backtest(trades, equity)
        rows.append({
            "cost_per_side": cost,
            "round_trip": 2 * cost,
            "hold": hold,
            "total_return": summary["total_return"],
            "avg_trade": summary["avg_trade"],
            "max_drawdown": summary["max_drawdown"],
        })

pd.DataFrame(rows).round(4)

,cost_per_side,round_trip,hold,total_return,avg_trade,max_drawdown
0,0.0000,0.000,5,1.5354,0.0326,-0.3315
1,0.0000,0.000,10,2.1114,0.0578,-0.3423
2,0.0015,0.003,5,1.3033,0.0295,-0.3414
3,0.0015,0.003,10,1.8952,0.0546,-0.3443
4,0.0030,0.006,5,1.0925,0.0264,-0.3512
5,0.0030,0.006,10,1.6941,0.0514,-0.3463


In [5]:
tables = []
for hold in [5, 10]:
    trades, _ = run_backtest(dev, setup_flag(dev, "continuation"), hold, COST)
    signal_days = dev.index[dev.index.get_indexer(trades["entry_date"]) - 1]
    trades["trend"] = dev.loc[signal_days, "trend"].to_numpy()
    trades["vol_regime"] = dev.loc[signal_days, "vol_regime"].to_numpy()

    for regime in ["trend", "vol_regime"]:
        table = trades.groupby(regime)["net_return"].agg(
            trades="size",
            avg_trade="mean",
            win_rate=lambda r: (r > 0).mean(),
        )
        table.index = [f"{hold}d | {regime}={v}" for v in table.index]
        tables.append(table)

pd.concat(tables).round(4)

,trades,avg_trade,win_rate
5d | trend=range,2,0.0482,1.0000
5d | trend=up,30,0.0283,0.7333
5d | vol_regime=high,14,-0.0079,0.6429
5d | vol_regime=low,14,0.0603,0.8571
10d | trend=range,2,0.0433,0.5000
10d | trend=up,22,0.0556,0.5909
10d | vol_regime=high,12,-0.0126,0.5000
10d | vol_regime=low,9,0.1003,0.6667


### Held-out backtest: hypotheses and predictions (written before running)

**Primary (pre-registered in Phase 5):** continuation setup, 5- and 10-day holds, costs 0.30% per round trip.
- Prediction: [positive / negative] average trade after costs
- Prediction vs random up-trend trades: [better than ~95% of draws / not distinguishable]
- Prediction vs buy-and-hold: [lower / higher] max drawdown; [lower / higher] total return

**Exploratory (registered after development results):** continuation trades signalled in low volatility outperform those signalled in high volatility.
- Prediction: [holds / does not hold]

**Control:** pullback setup expected to show no edge after costs.

In [6]:
test = df.loc["2024-01-01":]
print("test:", test.index[0].date(), "->", test.index[-1].date(), "|", len(test), "days")

# 1. All strategies, same costs and rules as development
rows = []
for setup in ["continuation", "pullback"]:
    for hold in [5, 10]:
        trades, equity = run_backtest(test, setup_flag(test, setup), hold, COST)
        rows.append({"strategy": f"{setup} {hold}d", **summarize_backtest(trades, equity)})

bh_signal = pd.Series(False, index=test.index)
bh_signal.iloc[0] = True
trades, equity = run_backtest(test, bh_signal, len(test) - 2, COST)
rows.append({"strategy": "buy and hold", **summarize_backtest(trades, equity)})
test_results = pd.DataFrame(rows).set_index("strategy")

# 2. Random-entry baseline for the continuation strategy
rng = np.random.default_rng(1)
rand_rows = []
for hold in [5, 10]:
    trades, _ = run_backtest(test, setup_flag(test, "continuation"), hold, COST)
    actual, k = trades["net_return"].mean(), len(trades)
    all_days = net_trade_returns(test, hold, COST)
    up_days = all_days[test.loc[all_days.index, "trend"] == "up"]
    for label, pool in [("any day", all_days), ("up-trend day", up_days)]:
        sims = np.array([
            rng.choice(pool.to_numpy(), size=k, replace=False).mean()
            for _ in range(2000)
        ])
        rand_rows.append({
            "hold": hold,
            "random pool": label,
            "trades": k,
            "actual_avg_trade": actual,
            "random_median": np.median(sims),
            "share_random_beating_actual": (sims >= actual).mean(),
        })
test_random = pd.DataFrame(rand_rows)

# 3. Exploratory hypothesis: continuation trades by volatility regime at the signal
vol_tables = []
for hold in [5, 10]:
    trades, _ = run_backtest(test, setup_flag(test, "continuation"), hold, COST)
    signal_days = test.index[test.index.get_indexer(trades["entry_date"]) - 1]
    trades["vol_regime"] = test.loc[signal_days, "vol_regime"].to_numpy()
    table = trades.groupby("vol_regime")["net_return"].agg(
        trades="size",
        avg_trade="mean",
        win_rate=lambda r: (r > 0).mean(),
    )
    table.index = [f"{hold}d | vol_regime={v}" for v in table.index]
    vol_tables.append(table)
test_by_vol = pd.concat(vol_tables)

print(test_results.round(3))
print(test_random.round(4))
print(test_by_vol.round(4))

test: 2024-01-01 -> 2026-10-16 | 1020 days
                  total_return  annual_volatility  max_drawdown  trades  \
strategy                                                                  
continuation 5d          0.373              0.158        -0.112      12   
continuation 10d         0.646              0.188        -0.148      11   
pullback 5d              0.171              0.205        -0.240      40   
pullback 10d            -0.038              0.269        -0.426      27   
buy and hold             0.856              0.470        -0.530       1   

                  time_in_market  win_rate  avg_trade  
strategy                                               
continuation 5d            0.059     0.667      0.028  
continuation 10d           0.108     0.818      0.049  
pullback 5d                0.196     0.450      0.006  
pullback 10d               0.265     0.370      0.001  
buy and hold               0.999     1.000      0.856  
   hold   random pool  trades  actual_a

In [7]:
print(test_random.round(4).to_string())

   hold   random pool  trades  actual_avg_trade  random_median  share_random_beating_actual
0     5       any day      12            0.0283         0.0003                       0.0405
1     5  up-trend day      12            0.0283         0.0027                       0.0635
2    10       any day      11            0.0488         0.0056                       0.0420
3    10  up-trend day      11            0.0488         0.0084                       0.0630


## Phase 6 findings: backtesting the continuation setup (BTC/USDT)

**Assumptions:** signal at the daily close, entry at the next day's open, fixed holding period of 5 or 10 days with exit at that day's open; one position at a time, all-in, no leverage; signals during an open trade ignored; costs of 0.15% per side (0.10% fee + 0.05% slippage), i.e. 0.30% per round trip. Development period to 2023-12-31; held-out test 2024-01-01 to 2026-10-16, run once with predictions recorded beforehand.

**Observation (development):** continuation 5d / 10d: +130% / +190% total, average trade +3.0% / +5.5% after costs, maximum drawdown −34%, 7% / 11% of time in the market; buy-and-hold +416% with a −83% drawdown. Growth per unit of volatility 0.77 / 0.70 versus 0.42 for buy-and-hold. Against random trades taken on up-trend days, 16% / 20% of random draws did as well. Doubling costs to 0.60% per round trip reduced the average trade by about 0.3 percentage points. Pullback control: no edge after costs.

**Observation (held-out test):** continuation 5d / 10d: +37% / +65% total, average trade +2.8% / +4.9% after costs, maximum drawdown −11% / −15%, 12 / 11 trades; buy-and-hold +86% with a −53% drawdown. Growth per unit of volatility 0.76 / 1.04 versus 0.53. Against random up-trend trades, 6.4% / 6.3% of draws did as well. Pullback control: no edge after costs.

**Exploratory hypothesis (registered after development results):** low-volatility signals outperform high-volatility ones. Development: +6.0% versus −0.8% (5d). Test: +2.8% versus +2.8% (5d) and +4.6% versus +5.8% (10d). Not supported.

**Interpretation:** The continuation strategy survived realistic costs and remained profitable on untouched data, with a much better return per unit of risk and far smaller drawdowns than buy-and-hold, though a lower total return because it is rarely invested. Whether its edge goes beyond simply trading during uptrends is suggestive but not established: it was not distinguishable from random up-trend trades in development and was borderline in the test. The development volatility split did not replicate, which confirms the decision not to add it as a filter.

**Limitations:** BTC only; 32 / 24 development trades and 12 / 11 test trades; a single held-out period; costs are assumptions that should be checked against the current fee schedule; random baselines ignore the no-overlap rule; no position sizing beyond all-in.